# Chapter 09 — LLM 기반 로그 해석 (실습)

> **Chapter 09 · Part II**

Claude API 로 로그를 분류하고, mock 모드로도 끝까지 동작합니다.

> **주의**: 실제 Claude API 호출에는 API key 가 필요하다 (https://console.anthropic.com).
> 키가 있으면 실제 호출, 없으면 규칙 기반 mock 으로 끝까지 진행된다. mock 경로는 결정론이라
> 분류 결과를 재현할 수 있지만, **토큰·캐시 사용량 측정은 실제 호출에서만 가능하다.**

## 실습 목표
1. Claude API 첫 호출 — 응답 본문과 `usage` 토큰·비용 확인
2. 시스템 프롬프트 + JSON 구조화 출력
3. 정답 라벨이 붙은 합성 로그 50 건 일괄 분류 + 혼동 행렬 채점
4. ML 1 차 + LLM 2 차 하이브리드 — 전량 LLM 대비 LLM 호출 수·정확도 비교
5. Prompt caching 실측 — 모델별 최소 캐시 길이 조건과 비용 계산

In [1]:
# 첫 사용시에만 주석 해제하여 실행 (Colab/venv 환경에 따라 일부는 이미 설치됨)
# !pip install -q anthropic
# !pip install -q pandas
# !pip install -q numpy

In [2]:
import os, json, re, time
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

API_KEY = os.environ.get('ANTHROPIC_API_KEY')  # Colab Secrets 또는 환경변수
USE_REAL_API = bool(API_KEY)
MODEL = 'claude-haiku-4-5'   # 대량 1차 분류용. thinking 기본 꺼짐
PRICE_IN, PRICE_OUT = 1.0, 5.0        # Haiku 4.5 — 입력 $1 / 출력 $5 per MTok (본문 §9.3.1)

if USE_REAL_API:
    import anthropic
    client = anthropic.Anthropic(api_key=API_KEY)
    print(f'Real API mode — {MODEL}')
else:
    client = None
    print('Mock mode (API key 없음) — 분류는 규칙 기반 mock, 토큰·캐시 측정은 건너뛴다')

Real API mode — claude-haiku-4-5


---
## Step 0. 첫 호출과 usage

`resp.usage` 는 이 호출이 실제로 쓴 토큰을 알려 준다. 비용 계산도 추정이 아니라 이 값에서 시작한다.
응답 앞에 `thinking` 블록이 올 수 있으므로 `content[0]` 을 가정하지 않고 `text` 블록만 모은다.

In [3]:
def text_of(resp):
    """thinking 블록이 먼저 올 수 있으므로 text 블록만 모은다."""
    return ''.join(b.text for b in resp.content if b.type == 'text')

if USE_REAL_API:
    resp = client.messages.create(
        model=MODEL,
        max_tokens=256,
        messages=[{'role': 'user',
                   'content': '웹 접근 로그에서 status 401 이 뜻하는 바를 한 문장으로 설명하라.'}],
    )
    print(text_of(resp))
    u = resp.usage
    cost = u.input_tokens / 1e6 * PRICE_IN + u.output_tokens / 1e6 * PRICE_OUT
    print(f'\nusage: input={u.input_tokens} output={u.output_tokens} -> ${cost:.6f}')
else:
    print('mock 모드: 실제 호출과 usage 측정은 API 키가 있을 때만 가능하다.')

# HTTP Status 401 Unauthorized

**인증되지 않은 사용자가 보호된 자원에 접근하려고 했음을 의미합니다.**

(즉, 유효한 인증 정보(로그인, API 키 등)가 없거나 잘못되어 요청이 거부된 상태입니다.)

usage: input=46 output=102 -> $0.000556


---
## Step 1. 시스템 프롬프트 설정

판정 규칙과 **로그 필드의 의미**를 함께 적는다. 필드 의미를 알려 주지 않으면
`fail_count_5m` 같은 맥락 필드가 있어도 모델이 한 줄만 보고 판단하게 된다.

In [4]:
SYSTEM_PROMPT = '''당신은 SOC 의 senior security analyst 다.
입력 로그를 분석하고 다음 JSON 으로만 응답한다.
{
  "summary": "한 문장 요약",
  "threat_category": "BENIGN|SQLi|XSS|DDoS|PortScan|Brute|Other",
  "severity": 1-5 정수,
  "indicators": ["근거 필드"],
  "recommended_actions": ["대응 1~3"]
}
로그 필드의 의미:
- url, ua, status: 요청 경로, User-Agent, 응답 코드
- fail_count_5m: 같은 출발지 IP 의 최근 5 분 인증 실패 횟수
판정 규칙:
1. fail_count_5m 이 10 이상이면서 인증 실패가 반복되면 Brute
2. 경로 조작(../, %2e%2e%2f) 은 Other
3. 확신 없으면 BENIGN, severity=1
4. 로그에 없는 IP·도메인을 만들지 말 것 (환각 금지)'''
print(f'system prompt length = {len(SYSTEM_PROMPT)} chars')

system prompt length = 494 chars


---
## Step 2. 정답 라벨이 붙은 합성 로그 50 건

채점하려면 정답이 있어야 한다. 라벨은 로그 딕셔너리 **밖**의 리스트에 따로 두어
모델 입력에 정답이 섞여 들어가지 않게 한다. Brute 로 라벨한 로그에는 판정 근거가 되는
`fail_count_5m` 이 실제로 들어 있다 — 한 줄만 보고는 알 수 없는 것을 정답으로 삼지 않는다.
정상 30 건에는 `select` 나 `sql-injection` 같은 단어가 URL 에 들어 있는 경계 로그 2 건을 섞었다.
키워드만 보는 탐지기가 오탐하기 쉬운 자리다. 모델이 이 두 건을 어떻게 판정했는지는
혼동 행렬의 BENIGN 행에서 확인한다.

In [5]:
rng = np.random.default_rng(42)
BENIGN_URLS = ['/index.html', '/products', '/api/v1/users', '/static/app.js', '/health']
ATTACK_URLS = [("/login?id=' OR 1=1--", 'SQLi'),
               ('/item?id=1 UNION SELECT NULL,NULL--', 'SQLi'),
               ("/item?id=1' OR '1'='1", 'SQLi'),
               ('/report?year=2026 UNION SELECT password FROM users--', 'SQLi'),
               ('/search?q=<script>alert(1)</script>', 'XSS'),
               ('/comment?text=<script>document.cookie</script>', 'XSS'),
               ('/profile?name=<script>alert(document.domain)</script>', 'XSS'),
               ('/download?file=../../etc/passwd', 'Other'),
               ('/view?f=../../../windows/win.ini', 'Other'),
               ('/backup?path=../../var/log/auth.log', 'Other')]

# 공격처럼 보이지만 정상인 경계 로그 — 키워드만 보는 탐지기가 오탐하는 자리
TRICKY_BENIGN = ['/search?q=select a plan for 2026', '/blog/how-to-prevent-sql-injection']

logs, labels = [], []
for i in range(28):                                    # 평범한 정상 28
    logs.append({'src': f'10.0.0.{int(rng.integers(1, 50))}', 'method': 'GET',
                 'url': BENIGN_URLS[i % len(BENIGN_URLS)], 'ua': 'Mozilla/5.0',
                 'status': 200, 'fail_count_5m': 0})
    labels.append('BENIGN')
for url in TRICKY_BENIGN:                              # 경계 정상 2
    logs.append({'src': '10.0.0.7', 'method': 'GET', 'url': url,
                 'ua': 'Mozilla/5.0', 'status': 200, 'fail_count_5m': 0})
    labels.append('BENIGN')
for i in range(10):                                    # 페이로드 공격 10 (SQLi 4 · XSS 3 · 경로 조작 3)
    url, lab = ATTACK_URLS[i]
    logs.append({'src': '10.0.0.99', 'method': 'POST', 'url': url,
                 'ua': str(rng.choice(['sqlmap/1.5', 'nikto/2.1', 'curl/7.0'])),
                 'status': int(rng.choice([401, 403, 500])), 'fail_count_5m': 0})
    labels.append(lab)
for i in range(10):                                    # 무차별 대입 10
    logs.append({'src': f'192.168.1.{100 + i}', 'method': 'POST', 'url': '/login',
                 'ua': 'Mozilla/5.0', 'status': 401,
                 'fail_count_5m': int(rng.integers(18, 60))})
    labels.append('Brute')

print(f'총 {len(logs)} 건 — 정답 라벨 분포')
print(pd.Series(labels).value_counts().to_string())
logs[0], labels[0]

총 50 건 — 정답 라벨 분포
BENIGN    30
Brute     10
SQLi       4
XSS        3
Other      3


({'src': '10.0.0.5',
  'method': 'GET',
  'url': '/index.html',
  'ua': 'Mozilla/5.0',
  'status': 200,
  'fail_count_5m': 0},
 'BENIGN')

---
## Step 3. 분류 함수 — 실제 API 또는 Mock

In [6]:
def mock_classify(log: dict) -> dict:
    """API 키 없을 때 쓰는 결정론 규칙 mock. LLM 을 흉내 낼 뿐 정답 오라클이 아니다."""
    url = log.get('url', '')
    if re.search(r'union\s+select|or\s+1=1', url, re.I):
        return {'summary': 'SQL 구문이 파라미터에 포함', 'threat_category': 'SQLi', 'severity': 4,
                'indicators': ['url SQL keyword'], 'recommended_actions': ['src 차단']}
    if re.search(r'<script>|javascript:', url, re.I):
        return {'summary': '스크립트 태그 삽입 시도', 'threat_category': 'XSS', 'severity': 4,
                'indicators': ['url script'], 'recommended_actions': ['src 차단']}
    if '../' in url or '%2e%2e%2f' in url.lower():
        return {'summary': '경로 조작 시도', 'threat_category': 'Other', 'severity': 3,
                'indicators': ['url ../'], 'recommended_actions': ['WAF 규칙 추가']}
    if log.get('status') == 401 and log.get('fail_count_5m', 0) >= 10:
        return {'summary': '5 분 내 인증 실패 반복', 'threat_category': 'Brute', 'severity': 3,
                'indicators': ['fail_count_5m'], 'recommended_actions': ['계정 잠금']}
    return {'summary': '정상 트래픽', 'threat_category': 'BENIGN', 'severity': 1,
            'indicators': [], 'recommended_actions': []}

def real_classify(log: dict) -> dict:
    resp = client.messages.create(
        model=MODEL,
        max_tokens=512,
        system=SYSTEM_PROMPT,
        messages=[{'role': 'user', 'content': f'로그: {json.dumps(log, ensure_ascii=False)}'}],
    )
    text = ''.join(b.text for b in resp.content if b.type == 'text')
    usage = {'input': resp.usage.input_tokens, 'output': resp.usage.output_tokens}
    try:
        return {**json.loads(text), '_usage': usage}
    except json.JSONDecodeError:
        m = re.search(r'```json\s*(.*?)\s*```', text, re.DOTALL)
        if m:
            try:
                return {**json.loads(m.group(1)), '_usage': usage}
            except json.JSONDecodeError:
                pass
        # 파싱 실패를 BENIGN 으로 돌려주면 정상 로그에서 실패가 정답으로 채점된다
        return {'threat_category': None, 'parse_error': True, 'raw': text[:200], '_usage': usage}

classify = real_classify if USE_REAL_API else mock_classify

In [7]:
# 50 건 일괄 분류 (실제 API 는 병렬)
start = time.time()
if USE_REAL_API:
    with ThreadPoolExecutor(max_workers=5) as ex:
        results = list(ex.map(classify, logs))
else:
    results = [classify(log) for log in logs]
elapsed = time.time() - start

df = pd.DataFrame([{'url': lg['url'], 'ua': lg['ua'], 'status': lg['status'],
                    'fail_count_5m': lg['fail_count_5m'], 'truth': t,
                    'pred': r.get('threat_category'), 'parse_error': bool(r.get('parse_error'))}
                   for lg, t, r in zip(logs, labels, results)])
acc = (df['pred'] == df['truth']).mean()
tok = sum(r.get('_usage', {}).get('output', 0) for r in results)
print(f'분류 {len(df)} 건 / {elapsed:.1f}s / 정확도 {acc:.1%} / 파싱 실패 {int(df["parse_error"].sum())} 건')
if USE_REAL_API:
    tin = sum(r['_usage']['input'] for r in results)
    print(f'토큰: 입력 {tin} · 출력 {tok} -> ${tin/1e6*PRICE_IN + tok/1e6*PRICE_OUT:.4f}')

print('\n--- 혼동 행렬 (행=정답, 열=예측) ---')
print(pd.crosstab(df['truth'], df['pred'].fillna('PARSE_ERROR')).to_string())

if not USE_REAL_API:
    print('\nmock 규칙은 라벨을 만든 규칙과 같은 패턴이므로 정확도가 높게 나온다.'
          ' 이 숫자는 mock 의 성질이지 LLM 의 성능이 아니다.')

wrong = df[df['pred'] != df['truth']]
print(f'\n--- 오분류 {len(wrong)} 건 ---')
print(wrong[['url', 'ua', 'status', 'fail_count_5m', 'truth', 'pred']].to_string(index=False)
      if len(wrong) else '없음')

분류 50 건 / 20.0s / 정확도 100.0% / 파싱 실패 0 건
토큰: 입력 19650 · 출력 9055 -> $0.0649

--- 혼동 행렬 (행=정답, 열=예측) ---
pred    BENIGN  Brute  Other  SQLi  XSS
truth                                  
BENIGN      30      0      0     0    0
Brute        0     10      0     0    0
Other        0      0      3     0    0
SQLi         0      0      0     4    0
XSS          0      0      0     0    3

--- 오분류 0 건 ---
없음


---
## Step 4. ML 1 차 + LLM 2 차 하이브리드

ML 1 차는 명백한 정상·명백한 공격을 즉시 처리하고, 모호한 구간만 LLM 에 넘긴다.
여기서 확인할 것은 두 가지다 — **LLM 호출을 얼마나 줄이는가**, 그리고 **그 대가로 정확도를 얼마나 잃는가**.
ML 은 카테고리를 구분하지 못하므로 비교는 `공격 / 정상` 이진 판정으로 맞춘다.
Step 3 에서 이미 분류한 결과를 재사용하므로 같은 로그를 다시 호출하지 않는다.

In [8]:
def ml_score(log: dict) -> float:
    """ML 1차 — 단순 룰 기반 확률 mock."""
    url, ua = log.get('url', ''), log.get('ua', '')
    # '../' 를 정규식에 넣을 때 점을 이스케이프하지 않으면 '.' 가 임의 문자가 되어
    # /api/v1/users 의 'pi/' 까지 걸린다. 정상 로그가 즉시 차단으로 튀는 오탐이다.
    if re.search(r'union\s+select|or\s+1=1|<script>|javascript:|\.\./', url, re.I):
        return 0.95
    if re.search(r'sqlmap|nikto', ua, re.I):
        return 0.85
    if log.get('status') == 401:
        return 0.45
    return 0.05

fp = sorted({lg['url'] for lg, t in zip(logs, labels) if t == 'BENIGN' and ml_score(lg) > 0.7})
print(f'회귀 점검 — 정상인데 즉시 공격으로 튄 로그: {len(fp)} 건 {fp}')
assert not fp, '정상 로그가 ML 1차에서 즉시 차단됐다 (정규식 오탐)'

회귀 점검 — 정상인데 즉시 공격으로 튄 로그: 0 건 []


In [9]:
def to_binary(cat):
    if cat is None:
        return 'PARSE_ERROR'
    return 'BENIGN' if cat == 'BENIGN' else 'ATTACK'

rows = []
for lg, truth, llm in zip(logs, labels, results):   # Step 3 결과 재사용
    s = ml_score(lg)
    if s < 0.3:
        route, pred, calls = 'BENIGN-pass', 'BENIGN', 0
    elif s > 0.7:
        route, pred, calls = 'ATTACK-instant', 'ATTACK', 0
    else:
        route, pred, calls = 'LLM-2nd', to_binary(llm.get('threat_category')), 1
    rows.append({'url': lg['url'], 'fail_count_5m': lg['fail_count_5m'], 'ml': s, 'route': route,
                 'truth_bin': to_binary(truth), 'hybrid_pred': pred,
                 'llm_only_pred': to_binary(llm.get('threat_category')), 'llm_calls': calls})
routing = pd.DataFrame(rows)

print('--- 라우팅 분포 ---')
print(routing['route'].value_counts().to_string())

cmp = pd.DataFrame([
    {'전략': '전량 LLM (Step 3)', 'LLM 호출': len(logs),
     '이진 정확도': (routing['llm_only_pred'] == routing['truth_bin']).mean()},
    {'전략': 'ML 1차 + LLM 2차', 'LLM 호출': int(routing['llm_calls'].sum()),
     '이진 정확도': (routing['hybrid_pred'] == routing['truth_bin']).mean()},
])
cmp['호출 절감'] = (1 - cmp['LLM 호출'] / len(logs)).map('{:.0%}'.format)
cmp['이진 정확도'] = cmp['이진 정확도'].map('{:.1%}'.format)
print('\n--- 전량 LLM vs 하이브리드 ---')
print(cmp.to_string(index=False))

diff = routing[routing['hybrid_pred'] != routing['llm_only_pred']]
print(f'\n--- 두 전략의 판정이 갈린 로그 {len(diff)} 건 ---')
print(diff[['url', 'ml', 'route', 'truth_bin', 'llm_only_pred', 'hybrid_pred']].to_string(index=False)
      if len(diff) else '없음 (ML 1차가 거른 구간에서 LLM 과 같은 판정)')

--- 라우팅 분포 ---
route
BENIGN-pass       30
ATTACK-instant    10
LLM-2nd           10

--- 전량 LLM vs 하이브리드 ---
             전략  LLM 호출 이진 정확도 호출 절감
전량 LLM (Step 3)      50 100.0%    0%
 ML 1차 + LLM 2차      10 100.0%   80%

--- 두 전략의 판정이 갈린 로그 0 건 ---
없음 (ML 1차가 거른 구간에서 LLM 과 같은 판정)


---
## Step 5. Prompt caching 실측

`cache_control` 을 달았다고 캐시가 걸리는 것은 아니다. **캐시 가능한 프리픽스에는 모델별 최소
토큰 수가 있고, 그보다 짧으면 오류 없이 조용히 캐시되지 않는다.**

| 모델 | 최소 캐시 길이 |
|---|---|
| Opus 5 | 512 토큰 |
| Sonnet 5 | 1,024 토큰 |
| Haiku 4.5 | 4,096 토큰 |

(출처: Anthropic, "Prompt caching" — https://platform.claude.com/docs/en/build-with-claude/prompt-caching)

그래서 Step 1 의 300 자짜리 시스템 프롬프트에 `cache_control` 을 달면 Haiku 에서는 아무 일도
일어나지 않는다. 아래 셀은 **짧은 프롬프트**와 **최소 길이를 넘긴 긴 프롬프트**를 각각 두 번씩
호출해 `usage.cache_creation_input_tokens` / `cache_read_input_tokens` 로 직접 확인한다.

In [10]:
# 최소 길이를 넘기는 고정 프리픽스 — 실무에서는 사내 자산 인벤토리·탐지 규칙표처럼
# 요청마다 바뀌지 않는 참조 자료가 이 자리에 온다.
def asset_appendix(n_rows: int) -> str:
    lines = ['[사내 자산 인벤토리] 아래 표의 호스트만 정상 자산이다. 목록에 없는 IP 는 미등록 자산으로 본다.']
    for i in range(n_rows):
        lines.append(f'HOST-{i:04d} | 10.0.{i // 250}.{i % 250} | zone=dmz seg={i % 7} '
                     f'owner=team{i % 12} svc=http port=8{i % 10}80 patch=2026-0{i % 9 + 1}-15 '
                     f'criticality={i % 5 + 1} note=정기 점검 대상, 외부 노출 없음')
    return '\n'.join(lines)

MIN_CACHE_TOKENS = {'claude-opus-5': 512, 'claude-sonnet-5': 1024, 'claude-haiku-4-5': 4096}
MIN_FOR_MODEL = MIN_CACHE_TOKENS['claude-haiku-4-5']
SAMPLE_LOG = logs[30]   # 공격 로그 한 건

def count_tokens(system_blocks):
    r = client.messages.count_tokens(
        model=MODEL, system=system_blocks,
        messages=[{'role': 'user', 'content': f'로그: {json.dumps(SAMPLE_LOG, ensure_ascii=False)}'}])
    return r.input_tokens

def cached_system(text):
    return [{'type': 'text', 'text': text, 'cache_control': {'type': 'ephemeral'}}]

def call_once(system_blocks):
    resp = client.messages.create(
        model=MODEL, max_tokens=256, system=system_blocks,
        messages=[{'role': 'user', 'content': f'로그: {json.dumps(SAMPLE_LOG, ensure_ascii=False)}'}])
    u = resp.usage
    return {'input': u.input_tokens, 'output': u.output_tokens,
            'cache_write': getattr(u, 'cache_creation_input_tokens', 0) or 0,
            'cache_read': getattr(u, 'cache_read_input_tokens', 0) or 0}

if USE_REAL_API:
    # 최소 길이를 넘을 때까지 프리픽스를 늘린다 (count_tokens 는 과금되지 않는다)
    n = 40
    while True:
        long_system = SYSTEM_PROMPT + '\n\n' + asset_appendix(n)
        n_tok = count_tokens(cached_system(long_system))
        if n_tok > MIN_FOR_MODEL * 1.1 or n > 2000:
            break
        n = int(n * 1.6) + 10
    short_tok = count_tokens(cached_system(SYSTEM_PROMPT))
    print(f'짧은 프롬프트 = {short_tok} 토큰 (최소 {MIN_FOR_MODEL} 미만)')
    print(f'긴 프롬프트   = {n_tok} 토큰 (자산 {n} 행)\n')

    usage_rows = []
    for label, sysmsg in [('짧은 프롬프트', SYSTEM_PROMPT), ('긴 프롬프트', long_system)]:
        for attempt in ('1회차(기록)', '2회차(재사용)'):
            u = call_once(cached_system(sysmsg))
            usage_rows.append({'프롬프트': label, '호출': attempt, **u})
            time.sleep(1)
    usage_df = pd.DataFrame(usage_rows)
    print(usage_df.to_string(index=False))
    print('\ncache_write 와 cache_read 가 모두 0 이면 캐시가 걸리지 않은 것이다 (오류는 나지 않는다).')
else:
    usage_df = None
    print('mock 모드: 캐시·토큰 사용량은 실제 호출에서만 측정할 수 있다. 이 셀은 건너뛴다.')
    print('최소 캐시 길이(토큰):', MIN_CACHE_TOKENS)

짧은 프롬프트 = 399 토큰 (최소 4096 미만)
긴 프롬프트   = 5859 토큰 (자산 74 행)



   프롬프트       호출  input  output  cache_write  cache_read
짧은 프롬프트  1회차(기록)    399     256            0           0
짧은 프롬프트 2회차(재사용)    399     256            0           0
 긴 프롬프트  1회차(기록)     71     256         5788           0
 긴 프롬프트 2회차(재사용)     71     256            0        5788

cache_write 와 cache_read 가 모두 0 이면 캐시가 걸리지 않은 것이다 (오류는 나지 않는다).


In [11]:
# 비용 = 입력 × $1 + 캐시 기록 × $1×1.25 + 캐시 읽기 × $1×0.1 + 출력 × $5 (per MTok, 본문 §9.3.1·§9.4)
def call_cost(u):
    return (u['input'] * PRICE_IN + u['cache_write'] * PRICE_IN * 1.25
            + u['cache_read'] * PRICE_IN * 0.1 + u['output'] * PRICE_OUT) / 1e6

if USE_REAL_API:
    cost_df = usage_df.copy()
    cost_df['호출 비용($)'] = [call_cost(r) for r in usage_df.to_dict('records')]
    print(cost_df[['프롬프트', '호출', 'input', 'cache_write', 'cache_read', 'output', '호출 비용($)']]
          .to_string(index=False, float_format=lambda v: f'{v:.6f}'))

    print('\n--- 긴 프리픽스로 1 만 건을 분류할 때 (2 회차 실측값을 반복한다고 가정) ---')
    reuse = usage_df[(usage_df['프롬프트'] == '긴 프롬프트') & (usage_df['호출'] == '2회차(재사용)')].iloc[0].to_dict()
    write = usage_df[(usage_df['프롬프트'] == '긴 프롬프트') & (usage_df['호출'] == '1회차(기록)')].iloc[0].to_dict()
    no_cache = {'input': reuse['input'] + reuse['cache_read'], 'cache_write': 0, 'cache_read': 0,
                'output': reuse['output']}
    print(f"캐시 없이 1 만 건 : ${call_cost(no_cache) * 10000:.2f}")
    print(f"캐시 사용 1 만 건 : ${call_cost(write) + call_cost(reuse) * 9999:.2f}"
          '  (5 분 TTL 안에서 재사용된다고 가정)')
    print('\n캐시는 입력 토큰만 줄인다. 출력 비용은 그대로이므로 총액 절감률은 입력 절감률보다 항상 낮다.')
else:
    print('mock 모드: 실측 usage 가 없으므로 비용 비교를 건너뛴다.')
    print('키를 넣고 다시 실행하면 위 셀의 usage 값으로 같은 계산을 하게 된다.')

   프롬프트       호출  input  cache_write  cache_read  output  호출 비용($)
짧은 프롬프트  1회차(기록)    399            0           0     256  0.001679
짧은 프롬프트 2회차(재사용)    399            0           0     256  0.001679
 긴 프롬프트  1회차(기록)     71         5788           0     256  0.008586
 긴 프롬프트 2회차(재사용)     71            0        5788     256  0.001930

--- 긴 프리픽스로 1 만 건을 분류할 때 (2 회차 실측값을 반복한다고 가정) ---
캐시 없이 1 만 건 : $71.39
캐시 사용 1 만 건 : $19.30  (5 분 TTL 안에서 재사용된다고 가정)

캐시는 입력 토큰만 줄인다. 출력 비용은 그대로이므로 총액 절감률은 입력 절감률보다 항상 낮다.


---
## 정리
- LLM 한 번의 호출로 보안 로그의 분류·설명·대응 추천을 함께 받았습니다. `usage` 로 호출마다 쓴 토큰과 비용을 확인했습니다.
- 정답 라벨을 붙여 두면 일괄 분류 결과를 혼동 행렬로 채점할 수 있습니다. 어느 카테고리에서 틀리는지가 프롬프트를 고치는 출발점입니다.
- ML 이 명백한 케이스를 먼저 거르는 하이브리드는 LLM 호출 수를 크게 줄입니다. 호출 절감과 이진 정확도를 같은 표에서 비교했습니다.
- ML 1 차의 정규식은 자기 자신이 오탐 원인이 될 수 있습니다. `../` 의 점을 이스케이프하지 않으면 `/api/v1/users` 같은 정상 경로가 즉시 차단으로 튀므로, 정상 로그에 대한 회귀 점검을 넣었습니다.
- Prompt caching 은 프리픽스가 모델별 최소 길이(Haiku 4.5 는 4,096 토큰) 를 넘을 때만 걸리고, 짧으면 오류 없이 무시됩니다. `cache_creation_input_tokens`·`cache_read_input_tokens` 로 직접 확인해야 합니다.

> **다음 챕터:** Prompt Engineering 심화와 XAI — Role·Few-shot·CoT·구조화 출력 패턴.